In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [4]:
df = pd.read_excel("/Users/sidhi/Desktop/new_car/data/audi.xlsx")

In [5]:
df.head()

df.info()

df.describe()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10668 entries, 0 to 10667
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         10668 non-null  object 
 1   year          10668 non-null  int64  
 2   price         10668 non-null  int64  
 3   transmission  10668 non-null  object 
 4   mileage       10668 non-null  int64  
 5   fuelType      10668 non-null  object 
 6   tax           10668 non-null  int64  
 7   mpg           10668 non-null  float64
 8   engineSize    10668 non-null  float64
dtypes: float64(2), int64(4), object(3)
memory usage: 750.2+ KB


model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
dtype: int64

In [6]:
X = df.drop("price", axis=1)
y = df["price"]

In [7]:
categorical_cols = [
    'model',
    'transmission',
    'fuelType'
]

numerical_cols = [
    'year',
    'mileage',
    'tax',
    'mpg',
    'engineSize'
]

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_cols
        )
    ],
    remainder='passthrough'
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

In [11]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', rf)
])

In [12]:
model.fit(X_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['model', 'transmission',
                                                   'fuelType'])])),
                ('regressor',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [13]:
y_pred = model.predict(X_test)

In [14]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE : 1518.1436302639747
RMSE: 2290.3153890476037
R2 Score: 0.9652910608705657


In [15]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importances = model.named_steps["regressor"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df.sort_values(
    by="Importance",
    ascending=False,
    inplace=True
)

print(importance_df.head(10))

                     Feature  Importance
34            remainder__mpg    0.456866
31           remainder__year    0.214923
35     remainder__engineSize    0.152180
32        remainder__mileage    0.049870
33            remainder__tax    0.032507
13            cat__model_ R8    0.020250
26  cat__transmission_Manual    0.016822
11            cat__model_ Q7    0.009027
0             cat__model_ A1    0.006634
30      cat__fuelType_Petrol    0.005915


In [16]:
joblib.dump(model, "audi_price_model.pkl")

['audi_price_model.pkl']

In [17]:
model = joblib.load("audi_price_model.pkl")

In [18]:
sample = pd.DataFrame({
    "model": ["A4"],
    "year": [2018],
    "transmission": ["Automatic"],
    "mileage": [35000],
    "fuelType": ["Diesel"],
    "tax": [145],
    "mpg": [55.4],
    "engineSize": [2.0]
})

prediction = model.predict(sample)

print("Predicted Price:", prediction[0])

Predicted Price: 23169.515


In [28]:
import joblib

joblib.dump(model, "/Users/sidhi/Desktop/new_car/audi_price_model.pkl")

['/Users/sidhi/Desktop/new_car/audi_price_model.pkl']